In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
import pdfplumber
from collections import defaultdict
from bs4 import BeautifulSoup
import requests
import datetime
from selenium import webdriver
from time import sleep
import os
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import camelot
import urllib3
from openpyxl import load_workbook
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'KZ ARDFM' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running KZ ARDFM Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict={

         regulatorName + ' 1': 'https://www.gov.kz/memleket/entities/ardfm/activities/789?lang=en',
         regulatorName + ' 2': 'https://www.gov.kz/memleket/entities/ardfm/activities/789?lang=en',
         regulatorName + ' 3': 'https://www.gov.kz/memleket/entities/ardfm/activities/789?lang=en',
         regulatorName + ' 4': 'https://www.gov.kz/memleket/entities/ardfm/activities/847?lang=en',
        # # regulatorName + ' 5': 'https://publicreg.myafsa.com/fintech/',
         regulatorName + ' 6': 'https://www.gov.kz/memleket/entities/ardfm/activities/847?lang=en',
         regulatorName + ' 7': 'https://www.gov.kz/memleket/entities/ardfm/activities/788?lang=en',
         regulatorName + ' 8': 'https://www.gov.kz/memleket/entities/ardfm/activities/788?lang=en',
         regulatorName + ' 9': 'https://www.gov.kz/memleket/entities/ardfm/activities/16487?lang=en',
         regulatorName + ' 10': 'https://www.gov.kz/memleket/entities/ardfm/activities/16487?lang=en',
         regulatorName + ' 11': 'https://www.gov.kz/memleket/entities/ardfm/activities/16487?lang=en',

        }



Typology={

        regulatorName + ' 1': 'ORGANIZATIONS ENGAGED IN CERTAIN TYPES OF BANKING ОPERATIONS',
        regulatorName + ' 2': 'SECOND TIER BANKS',
        regulatorName + ' 3': 'CONSOLIDATED SUPERVISION',
        regulatorName + ' 4': 'PARTICIPANTS OF THE INSURANCE MARKET',
        # regulatorName + ' 5': 'https://publicreg.myafsa.com/fintech/',
        regulatorName + ' 6': 'CONSOLIDATED SUPERVISION',
        regulatorName + ' 7': 'PROFESSIONAL SECURITIES MARKET PARTICIPANTS',
        regulatorName + ' 8': 'CONSOLIDATED SUPERVISION',
        regulatorName + ' 9': 'CREDIT BUREAUS',
        regulatorName + ' 10': 'ORGANIZATIONS ENGAGED IN MICROFINANCE ACTIVITIES',
        regulatorName + ' 11': 'SERVICE COMPANIES',

        }

city_list = ['Almaty','Shymkent','Astana','Nur-Sultan','Qaraghandy','Öskemen','Aqtöbe','Taraz','Semey','Pavlodar','Türkistan','Qyzylorda','Oral','Qostanay','Petropavl','Temirtaū','Aqtaū','Kökshetaū','Ekibastuz','Zhangaözen','Atyraū','Taldyqorghan','Rudnyy','Zhezqazghan','Kentaū','Balqash','Aqsū','Sätbayev','Kapchagay','Qaskeleng','Ridder','Zyryanovsk','Talghar','Stepnogorsk','Shchūchīnsk','Qarabulaq','Zharkent','Qapshaghay','Soran','Ayagöz','Arys','Beyneū','Baikonur','Saryaghash','Shakhtīnsk','Qulsary','Esik','Zhetisay','Shū','Zhitiqara','Aksay','Sayram','Shiyeli','StaryBeyneu','Balyqshy','Tekeli','Uzynaghash','Qarataū','Aral','Atbasar','Oktyabrsk','Sortöbe','Arqalyq','Shalqar','Boralday','Shelek','Abay','Lengir','Üshtöbe','Sarykemer','Oytal','Qarazhal','Khromtaū','Mangghystaū','Zhangatas','OtegeBatyr','Alga','Atasū','TölBī','Shemonaīkha','Zhosaly','Uryzhar','Shongzhy','Makīnsk','Qarabulaq','Üsharal','Bayserke','Maqat','Qazyqurt','Zaysan','Sarqan','EskĪkan','Aqköl','Shamalgan','Masangshy','Shetpe','Īnderbor','Qulan','BalpyBī','Tuzdybastaū','Ereymentaū','BaūyrzhaMomyshuly','Taiynsha','Bayanaūyl','Zhetibay','Novoīshīmskīy','Esil','Derbisek','Emba','Abay','Lūgovoy','Serebryansk','Qarabalyq','Saūmalköl','Borovskoy','Sholaqqorghan','Qarqaraly','Kegen','Qashyr','Makhambet','Kürshim','Chapaev','Moyynqum','Būlaevo','Zerendi','Zhansügirov','Shar','Quryq','Qusmuryn','Osakarovka','Koktobe','Sharbaqty','Ertis','Narynqol','Qazaly','Bayghanīn','Amangeldi','Zhänibek','Sergeevka','Mamlyutka','Arshaly','Shyngghyrlaū','Mezhdūrechenskoye','Kishkeneköl','Saghyz','Fyodorovka','Astrakhan','Derzhavīnsk','Miyaly','Shortandy','Smīrnovo','Badamsha','Fort-Shevchenko','Yavlenka','Presnovka','Pavlodarskoe','Qaraoy','Torghay','Botaqara','Kaztalovka','Zhympīty','Dostyq','Zhaqsy','Aqsū-Ayuly','Qamysty','Balkashīno','Qaramendi','Stepnyak','Kūlagīno','Qosshaghyl','Sayqyn','Kökpekti','Lenīngradskoe','Qarasū','Qaratöbe','Myngbayev','Egindiköl','Taskesken','Aqtoghay','Ketpen','Aqsū','Myrzakent','Malīnovka','Koshi','Asa','Temirlan','Zatobyl','TuraRysqulov','Shornaq','Abay','Saryagash','Besköl','Energetīcheskīy','Molodezhnyy','Qorday','Ülbi','Merke','Taranovskoe','Lisakovsk','Shayan','Glūbokoe','Borodūlīkha','NovayShūl’ba','Mortyq','Zhangaqorghan','Kamenka','Saryözek','Äulīeköl','Yul’evka','Aryqbalyq','Vozvyshenka','Saryköl','Shäūildir','Uzynköl','Ganyushkīno','Shubarqudyq','Denīsovka','Obaghan','Tīmīryazevo','Samarskoe','Aqqystaū','ÜlkeVladīmīrovka','Oktyabr’skoe','Fedorovka','Talshyq','Zhelezīnka','Uil','Aqsūat','Georgīevka','Bol’shenarymskoe','Ūspenka','Zhangaqīma','Zhalaghash','Ulytaū','Aqqū','Terengözek','Aqtoghay','Dzhangala','Novoalekseyevka','Terekti','Egindiköl','Baqanas','Qorghalzhyn','Tasty-Taldy','Qaraūyl','Köktöbe','Komsomol’skoe','Kūrchatov','Kīevka','Yrghyz']

In [5]:
#------------------------------------------------ Function_ ----------------------------------------

def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


zip_pattern = re.compile(r"\b\d{6}\b")

def extract_zip(address: str):
    """Return the first 5-digit ZIP in the address, or None if none found."""
    match = zip_pattern.search(address)
    return match.group(0) if match else ''
def extract_city(address: str, cities=city_list):
    text = address.lower()
    for city in cities:  # longest first
        if re.search(rf"\b{re.escape(city.lower())}\b", text):
            return city
    return ''



def pick(df, row, *keys):
    cols = list(df.columns)
    cols_lower = {c.lower(): i for i, c in enumerate(cols)}

    """Return first non-nullish value for the given column keys (case-insensitive)."""
    for k in keys:
        idx = cols_lower.get(k.lower())
        if idx is None:
            continue
        val = row[idx]
        if pd.isna(val):
            continue
        sval = str(val).strip()
        if sval.lower() == "null" or sval == "":
            continue
        return sval
    return ""



In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    # if reg == regulatorName+' 1':
    print(reg)
    # gov.kz is a JS-rendered SPA - requests returns ~1KB shell, must use Selenium
    driver.get(regdict[reg])
    wait = WebDriverWait(driver, 20)
    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h2")))
    except Exception:
        pass
    sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    print("Status: 200 (selenium)")
    needle = Typology.get(reg)
    links = []
    if needle:
        # Typology labels are now <a> anchors with #mcetoc_... fragments; walk forward collecting entity links until next typology anchor
        all_labels = set(Typology.values())
        anchor = soup.find("a", string=lambda s: s and s.strip() == needle)
        if anchor:
            for el in anchor.find_all_next():
                if el.name == "a":
                    text = el.get_text(strip=True)
                    if text in all_labels and text != needle:
                        break
                    href = el.get("href", "")
                    if href and not href.startswith("#") and ("financial-organizations" in href or "documents/details" in href):
                        if href.startswith("/"):
                            href = "https://www.gov.kz" + href
                        links.append(href)
    print(f"  found {len(links)} links")

    if reg == 'KZ ARDFM 1' or reg == 'KZ ARDFM 3' or reg =='KZ ARDFM 4'or reg =='KZ ARDFM 6'or reg =='KZ ARDFM 7'  or reg =='KZ ARDFM 9' or reg =='KZ ARDFM 11'  :
        for link in links:
            if '1059' in link or '43' in link or '27' in link or '728379' in link:
                continue
            driver.get(link)
            # wait until the Export to Excel button is clickable, then click
            try:
                btn = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//button[.//span[normalize-space()='Export to Excel']]"))
                )
                btn.click()
                sleep(3)
            except:
                print('Download Excel File does not Shoow up')
                continue

            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                sleep(3)
                print('[INFO] -- The download file is ready --')

            else:
                print('[INFO] -- Maybe the file still download, waiting.... -- ')
                sleep(6)
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]


            # Load the workbook
            try:
                workbook = load_workbook(filename=dl_files[0])
            except:
                from openpyxl.worksheet import _reader
                orig = _reader._cast_number
                def safe_cast(v):
                    try: return orig(v)
                    except ValueError: return v
                _reader._cast_number = safe_cast

                
                workbook = load_workbook(filename=dl_files[0])

            # Access the first sheet
            sheet = workbook.active
            # first row values as header
            first_row = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))
            headers = list(first_row)

            # data rows
            data_rows = sheet.iter_rows(min_row=2, values_only=True)

            df = pd.DataFrame(data_rows, columns=headers)
            df = df.fillna('').replace('null','')


            for _, item_ in df.iterrows():
                # print(item_['Name'])
                # print(item_['BIN'])
                # print(item_.contains.lower['address'])
                # print(item_['Phone number'])
                # print(item_['E-mail'])
                # print(item_['Web-site'])
                # print(item_['License'])
                name    = pick(df,item_, "name")
                bin_    = pick(df,item_, "bin")
                addr    = pick(df,item_, "address", "address (city, street, house, apartment)")
                phone   = pick(df,item_, "phone number")
                email   = pick(df,item_, "e-mail", "email")
                website = pick(df,item_, "web-site", "website")
                lic     = pick(df,item_, "license")
                zip_ = extract_zip(addr)
                city_ = extract_city(addr)
                print(name, bin_, addr, zip_, city_,phone, email, website, lic, sep=" | ")
                
                sqldict['Name'].append(name)
                sqldict['Address_1'].append(addr)
                sqldict['Phone'].append(phone)
                sqldict['Email'].append(email)
                sqldict['Website'].append(website)
                sqldict['Zip'].append(zip_)
                sqldict['City'].append(city_)
                sqldict['InternalID_1'].append(bin_)
                sqldict['InternalID_1_type'].append('BIN')
                sqldict['InternalID_2'].append(lic)
                sqldict['InternalID_2_type'].append('License')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("KZ")  
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)

            workbook.close()
            for _ in range(5):
                try:
                    if os.path.exists(tempfolder):
                        for rem in os.listdir(tempfolder):
                            os.remove(os.path.join(tempfolder, rem))
                    else:
                        os.mkdir(tempfolder)
                    break
                except PermissionError:
                    sleep(3)

    elif reg == 'KZ ARDFM 2':
        for link in links:
            if '18893' in link:
                continue
            driver.get(link)
            # wait until the Export to Excel button is clickable, then click
            try:
                btn = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//button[.//span[normalize-space()='Export to Excel']]"))
                )
                btn.click()
                sleep(3)
            except:
                print('Download Excel File does not Shoow up')
                # Click button with text "Download files"
                btn1 = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "//button[normalize-space()='Download files']")))
                btn1.click()

                # Then click button with text "Download"
                btn2 = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "//button[normalize-space()='Download']")))
                btn2.click()

            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                sleep(3)
                print('[INFO] -- The download file is ready --')

            else:
                print('[INFO] -- Maybe the file still download, waiting.... -- ')
                sleep(6)
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if dl_files[0].endswith('.xlsx'):
            # Load the workbook
                try:
                    workbook = load_workbook(filename=dl_files[0])
                except:
                    from openpyxl.worksheet import _reader
                    orig = _reader._cast_number
                    def safe_cast(v):
                        try: return orig(v)
                        except ValueError: return v
                    _reader._cast_number = safe_cast

                    
                    workbook = load_workbook(filename=dl_files[0])

                # Access the first sheet
                sheet = workbook.active
                # first row values as header
                first_row = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))
                headers = list(first_row)

                # data rows
                data_rows = sheet.iter_rows(min_row=2, values_only=True)

                df = pd.DataFrame(data_rows, columns=headers)
                df = df.fillna('').replace('null','')


                for _, item_ in df.iterrows():
                    # print(item_['Name'])
                    # print(item_['BIN'])
                    # print(item_.contains.lower['address'])
                    # print(item_['Phone number'])
                    # print(item_['E-mail'])
                    # print(item_['Web-site'])
                    # print(item_['License'])
                    name    = pick(df,item_, "name")
                    bin_    = pick(df,item_, "bin")
                    addr    = pick(df,item_, "address", "address (city, street, house, apartment)")
                    phone   = pick(df,item_, "phone number")
                    email   = pick(df,item_, "e-mail", "email")
                    website = pick(df,item_, "web-site", "website")
                    lic     = pick(df,item_, "license")
                    zip_ = extract_zip(addr)
                    city_ = extract_city(addr)
                    # print(name, bin_, addr, zip_, city_,phone, email, website, lic, sep=" | ")
                    
                    sqldict['Name'].append(name)
                    sqldict['Address_1'].append(addr)
                    sqldict['Phone'].append(phone)
                    sqldict['Email'].append(email)
                    sqldict['Website'].append(website)
                    sqldict['Zip'].append(zip_)
                    sqldict['City'].append(city_)
                    sqldict['InternalID_1'].append(bin_)
                    sqldict['InternalID_1_type'].append('BIN')
                    sqldict['InternalID_2'].append(lic)
                    sqldict['InternalID_2_type'].append('License')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict["Cntry"].append("KZ")  
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)

                workbook.close()
                for _ in range(5):
                    try:
                        if os.path.exists(tempfolder):
                            for rem in os.listdir(tempfolder):
                                os.remove(os.path.join(tempfolder, rem))

                        break
                    except PermissionError:
                        sleep(3)

            elif dl_files[0].endswith('.pdf'):
                if '18923' in link:
                    col_edges = [0,72,212,300,400,660,1000]  # <-- set these to your measured x’s
                    row_tol = 15  # vertical tolerance; increase to merge close lines, decrease to split    
                    def col_index(x_center):
                        for i in range(len(col_edges) - 1):
                            if col_edges[i] <= x_center < col_edges[i+1]:
                                return i
                        return None

                    with pdfplumber.open( dl_files[0]) as pdf:
                        page = pdf.pages[0]
                        words = page.extract_words()  # has x0, x1, top, bottom, text

                    # bucket words into lines using y (top/bottom); y grows upward in pdfplumber
                    lines = defaultdict(list)
                    for w in words:
                        mid_y = (w["top"] + w["bottom"]) / 2
                        key = round(mid_y / row_tol)
                        lines[key].append(w)

                    # order lines top→bottom
                    ordered_keys = sorted(lines, key=lambda k: min(w["top"] for w in lines[k]))

                    rows = []
                    for k in ordered_keys:
                        cells = [""] * (len(col_edges) - 1)
                        for w in sorted(lines[k], key=lambda w: w["x0"]):
                            x_mid = (w["x0"] + w["x1"]) / 2
                            ci = col_index(x_mid)
                            if ci is not None:
                                cells[ci] = (cells[ci] + " " + w["text"]).strip()
                        rows.append(cells)

                    df = pd.DataFrame(rows)  # or add column names if you have them
                    df = df.fillna("")  # ensure strings
                    name_col = 1 if 1 in df.columns else "Name"  # adjust if needed
                    quote_pat = re.compile(r"^\s*[‘']")

                    starts = df.index[df[name_col].str.match(quote_pat, na=False)].tolist()
                    starts.append(len(df))  # sentinel

                    rows = []
                    for i in range(len(starts) - 1):
                        lo, hi = starts[i], starts[i+1]
                        block = df.loc[lo:hi-1]
                        merged = {}
                        for c in df.columns:
                            texts = block[c].astype(str)
                            texts = [t.strip() for t in texts if t.strip()]  # drop empties
                            merged[c] = " ".join(texts)
                        rows.append(merged)

                    out = pd.DataFrame(rows)
                    for _, item_ in out.iterrows():
                        name_ = item_[out.columns[1]]
                        name_of_representative = item_[out.columns[2]]
                        address_ = item_[out.columns[4]]
                        phone_ = item_[out.columns[5]]

                        sqldict['Name'].append(name_)
                        sqldict['Name - Mother Company'].append(name_of_representative)
                        sqldict['Address_1'].append(address_)
                        sqldict['Phone'].append(phone_)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict["Cntry"].append("KZ")  
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)
                else:
                    tables = camelot.read_pdf(dl_files[0],pages="all",   flavor='stream',row_tol=10)
                    df_all = pd.concat([t.df for t in tables], ignore_index=True)
                    df_all = df_all.fillna("")
                    num_pat = re.compile(r"^\s*\d+")  # starts with a number
                    idx_starts = df_all.index[df_all[0].astype(str).str.match(num_pat, na=False)].tolist()
                    idx_starts.append(len(df_all))  # sentinel

                    merged_rows = []
                    for i in range(len(idx_starts) - 1):
                        lo, hi = idx_starts[i], idx_starts[i+1]
                        block = df_all.loc[lo:hi-1]
                        merged = {}
                        for c in df_all.columns:
                            vals = [v.strip() for v in block[c].astype(str) if v.strip()]
                            merged[c] = " ".join(vals)
                        merged_rows.append(merged)

                    out = pd.DataFrame(merged_rows)
                    for _, item_ in out.iterrows():
                        name_ = item_[out.columns[1]]
                        name_of_representative = item_[out.columns[2]]
                        id_data = item_[out.columns[4]]
                        address_ = item_[out.columns[5]]
                        # print(name_.replace('Representative Office','').strip())
                        # print(address_)
                        #phone_ = item_[out.columns[6].split('\n')]
                        #print(name_,name_of_representative,id_data,address_,phone_)

                        sqldict['Name'].append(name_.replace('Representative Office','').strip())
                        sqldict['Name - Mother Company'].append(name_of_representative)
                        sqldict['Address_1'].append(address_)
                        sqldict['InternalID_1'].append(id_data)
                        sqldict['InternalID_1_type'].append('Date of registration in Ministry of justice')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict["Cntry"].append("KZ")  
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)
                                    
                for _ in range(5):
                    try:
                        if os.path.exists(tempfolder):
                            for rem in os.listdir(tempfolder):
                                os.remove(os.path.join(tempfolder, rem))
                        break
                    except PermissionError:
                        sleep(3)
    
    elif reg =='KZ ARDFM 8':
        for link in links:
            driver.get(link)
            # wait until the Export to Excel button is clickable, then click
            try:
                btn = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//button[.//span[normalize-space()='Export to Excel']]"))
                )
                btn.click()
                sleep(3)
            except:
                print('Download Excel File does not Shoow up')
                continue

            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                sleep(3)
                print('[INFO] -- The download file is ready --')

            else:
                print('[INFO] -- Maybe the file still download, waiting.... -- ')
                sleep(6)
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]


            # Load the workbook
            try:
                workbook = load_workbook(filename=dl_files[0])
            except:
                from openpyxl.worksheet import _reader
                orig = _reader._cast_number
                def safe_cast(v):
                    try: return orig(v)
                    except ValueError: return v
                _reader._cast_number = safe_cast

                
                workbook = load_workbook(filename=dl_files[0])

            # Access the first sheet
            sheet = workbook.active
            # first row values as header
            first_row = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))
            headers = list(first_row)

            # data rows
            data_rows = sheet.iter_rows(min_row=2, values_only=True)

            df = pd.DataFrame(data_rows, columns=headers)
            df = df.fillna('').replace('null','')


            for _, item_ in df.iterrows():
                # print(item_['Name'])
                # print(item_['BIN'])
                # print(item_.contains.lower['address'])
                # print(item_['Phone number'])
                # print(item_['E-mail'])
                # print(item_['Web-site'])
                # print(item_['License'])
                name    = pick(df,item_, "name")
                bin_    = pick(df,item_, "bin")
                addr    = pick(df,item_, "address", "address (city, street, house, apartment)")
                phone   = pick(df,item_, "phone number")
                email   = pick(df,item_, "e-mail", "email")
                website = pick(df,item_, "web-site", "website")
                lic     = pick(df,item_, "license")
                zip_ = extract_zip(addr)
                city_ = extract_city(addr)
                print(name, bin_, addr, zip_, city_,phone, email, website, lic, sep=" | ")
                
                sqldict['Name'].append(name)
                sqldict['Address_1'].append(addr)
                sqldict['Phone'].append(phone)
                sqldict['Email'].append(email)
                sqldict['Website'].append(website)
                sqldict['Zip'].append(zip_)
                sqldict['City'].append(city_)
                sqldict['InternalID_1'].append(bin_)
                sqldict['InternalID_1_type'].append('BIN')
                sqldict['InternalID_2'].append(lic)
                sqldict['InternalID_2_type'].append('License')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("KZ")  
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)

            workbook.close()
            for _ in range(5):
                try:
                    if os.path.exists(tempfolder):
                        for rem in os.listdir(tempfolder):
                            os.remove(os.path.join(tempfolder, rem))
                    else:
                        os.mkdir(tempfolder)
                    break
                except PermissionError:
                    sleep(3)
    elif reg == 'KZ ARDFM 10':
        for link in links:
            driver.get(link)
            # wait until the Export to Excel button is clickable, then click
            try:
                btn = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//button[.//span[normalize-space()='Export to Excel']]"))
                )
                btn.click()
                sleep(3)
            except:
                print('Download Excel File does not Shoow up')
                # Click button with text "Download files"
                btn1 = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "//button[normalize-space()='Download files']")))
                btn1.click()
                sleep(3)

                # Then click button with text "Download"
                # wait for all Download buttons to exist
                buttons = wait.until(EC.presence_of_all_elements_located(
                    (By.XPATH, "//button[normalize-space()='Download']")
                ))
                sleep(2)
                last = buttons[-1]

                # ensure it’s visible/clickable; scroll into view
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", last)

                # try a normal click; fallback to JS if intercepted
                try:
                    last.click()
                except Exception:
                    driver.execute_script("arguments[0].click();", last)


            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                sleep(3)
                print('[INFO] -- The download file is ready --')

            else:
                print('[INFO] -- Maybe the file still download, waiting.... -- ')
                sleep(6)
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if dl_files[0].endswith('.xlsx'):
            # Load the workbook
                if '282950' in link:
                    data = pd.read_excel(dl_files[0])
                    data = data.dropna(subset=[data.columns[0]])
                    data.columns = data.iloc[0]
                    data = data.iloc[1:]
                    col = "Suspension of the licence for microfinance activities"
                    for index, row in data.iterrows():
                        susp = str(row.get(col, "")).strip().lower()
                        if susp in ("", "nan", "no", "false"):  # keep only when not suspended
                            name_ = row[data.columns[1]]
                            bin_ = row["BIN"]
                            licences = row[data.columns[3]]
                            date_of_issue = data.iloc[:, 4][index]
                            # order_number = data.iloc[:, 5][index]
                            topology = row[data.columns[-1]]
                            sqldict['Name'].append(name_)
                            sqldict['InternalID_1'].append(bin_)
                            sqldict['InternalID_1_type'].append('BIN')
                            sqldict['InternalID_2'].append(licences)
                            sqldict['InternalID_2_type'].append('License')
                            sqldict['RegulationDate'].append(date_of_issue)
                            sqldict['Typology'].append(topology)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegCtry'].append(reg.split(' ')[0])
                            sqldict['RegCode'].append(reg.split(' ')[1])
                            sqldict['ListCode'].append(reg.split(' ')[-1])
                            sqldict["Cntry"].append("KZ")  
                            sqldict['ListName'].append(Typology[reg])
                            sqldict['RegulationType'].append('Regulated')
                            sqldict = bourange_same_length_array(sqldict)
                else:
                    try:
                        workbook = load_workbook(filename=dl_files[0])
                    except:
                        from openpyxl.worksheet import _reader
                        orig = _reader._cast_number
                        def safe_cast(v):
                            try: return orig(v)
                            except ValueError: return v
                        _reader._cast_number = safe_cast
                        workbook = load_workbook(filename=dl_files[0])

                    # Access the first sheet
                    sheet = workbook.active
                    # first row values as header
                    first_row = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))
                    headers = list(first_row)

                    # data rows
                    data_rows = sheet.iter_rows(min_row=2, values_only=True)

                    df = pd.DataFrame(data_rows, columns=headers)
                    df = df.fillna('').replace('null','')


                    for _, item_ in df.iterrows():
                        # print(item_['Name'])
                        # print(item_['BIN'])
                        # print(item_.contains.lower['address'])
                        # print(item_['Phone number'])
                        # print(item_['E-mail'])
                        # print(item_['Web-site'])
                        # print(item_['License'])
                        name    = pick(df,item_, "name")
                        bin_    = pick(df,item_, "bin")
                        addr    = pick(df,item_, "address", "address (city, street, house, apartment)")
                        phone   = pick(df,item_, "phone number")
                        email   = pick(df,item_, "e-mail", "email")
                        website = pick(df,item_, "web-site", "website")
                        lic     = pick(df,item_, "license")
                        zip_ = extract_zip(addr)
                        city_ = extract_city(addr)
                        # print(name, bin_, addr, zip_, city_,phone, email, website, lic, sep=" | ")
                        
                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(addr)
                        sqldict['Phone'].append(phone)
                        sqldict['Email'].append(email)
                        sqldict['Website'].append(website)
                        sqldict['Zip'].append(zip_)
                        sqldict['City'].append(city_)
                        sqldict['InternalID_1'].append(bin_)
                        sqldict['InternalID_1_type'].append('BIN')
                        sqldict['InternalID_2'].append(lic)
                        sqldict['InternalID_2_type'].append('License')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict["Cntry"].append("KZ")  
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)

                    workbook.close()
                    for _ in range(5):
                        try:
                            if os.path.exists(tempfolder):
                                for rem in os.listdir(tempfolder):
                                    os.remove(os.path.join(tempfolder, rem))

                            break
                        except PermissionError:
                            sleep(3)

[INFO] : Working 1/1 _(KZ ARDFM 8)_ 
KZ ARDFM 8
Status: 200
[INFO] -- The download file is ready --
"Alatau City Bank"", JSC | 920140000084 | 242, Nursultan Nazarbayev street, Almaty, A26F8T9 |  | Almaty | +7 (717) 258-77-11 | info@jusan.kz | www.jusan.kz | 
"Bank CenterCredit", JSC | 980640000093 | 38, Al-Farabi Avenue, Almaty, A25D5G0 |  | Almaty | +7 (727) 259 85 98 | info@bcc.kz | www.bcc.kz | 
"Buzachi Neft" LLP | 931240001487 | 108A, Apt. 5, Al-Farabi Avenue, Almaty, Kazakhstan |  | Almaty |  |  |  | 
"Eurasian Financial Company", JSC |  | 59, Zheltoksan st., Almaty, 050004 | 050004 | Almaty | +7 (727) 259 87 17 |  |  | 
"ForteBank", JSC | 990740000683 | 8/1, Dostyk st., Astana |  | Astana | +7 (727) 258 75 75, +7 (7172) 58 75 75 | info@fortebank.com | www.fortebank.com | 
"Halyk Bank of Kazakhstan", JSC | 940140000385 | 40, Al-Farabi Ave., Almaty, А26М3К5 |  | Almaty | +7 (727) 259 07 77, +7 (727) 259 97 96 (техникалық көмек) | halykbank@halykbank.kz | www.halykbank.kz | 
"Nurba

C:\Users\wuj1\AppData\Local\Temp\8\ipykernel_21164\473336179.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  val = row[idx]


In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

#driver.quit()
sleep(3)

In [9]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,"""Alatau City Bank"""", JSC",920140000084,BIN,,License,...,,,,,,,,,,
1,,,,,,"""Bank CenterCredit"", JSC",980640000093,BIN,,License,...,,,,,,,,,,
2,,,,,,"""Buzachi Neft"" LLP",931240001487,BIN,,License,...,,,,,,,,,,
3,,,,,,"""Eurasian Financial Company"", JSC",,BIN,,License,...,,,,,,,,,,
4,,,,,,"""ForteBank"", JSC",990740000683,BIN,,License,...,,,,,,,,,,
5,,,,,,"""Halyk Bank of Kazakhstan"", JSC",940140000385,BIN,,License,...,,,,,,,,,,
6,,,,,,"""Nurbank"", JSC",930940000164,BIN,,License,...,,,,,,,,,,
7,,,,,,"""Orda Credit"" Credit partnership, LLP",,BIN,,License,...,,,,,,,,,,
8,,,,,,Abdrazakov E.S.,,BIN,,License,...,,,,,,,,,,
9,,,,,,Abguzhinov A.T.,,BIN,,License,...,,,,,,,,,,
